### Extraction des annotations Medkit vers un CSV tabulaire.

Ce script lit le fichier JSON produit par Medkit (medkit.json), récupère
les entités médicamenteuses et leurs attributs via les relations annotées,
puis exporte un fichier CSV exploitable pour la comparaison avec le gold standard.

In [1]:
import json
import pandas as pd
import itertools
from collections import defaultdict


# Fichiers d'entrée / sortie
INPUT_JSON = "medkit.json"
OUTPUT_CSV = "annot_medkit.csv"

# Labels utilisés dans le projet
MED_LABELS = {"Med", "Classe"}
DOSAGE_LABELS = {"Dosage"}
FREQ_LABELS = {"Freq"}
ROUTE_LABELS = {"Route"}
DATE_LABELS = {"Date"}
DATE_REL_LABELS = {"Date_relative", "Duree"}
COND_LABELS = {"Contexte"}

# Relations conservées pour reconstruire les liens entre entités
REL_LINK = {"Refer_to", "Start", "Stop", "Augmentation", "Duration_presc"}


def join_unique(values):
    "Concatène des valeurs en supprimant les doublons tout en gardant l'ordre."
    out = []
    for v in values:
        v = (v or "").strip()
        if v and v not in out:
            out.append(v)
    return " | ".join(out)



# Chargement du JSON Medkit
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

content = data["content"]
anns = content["anns"]


# Séparation des entités et des relations
ents_by_uid = {}
relations = []

for a in anns:
    cls = a.get("_class_name", "")

    # Extraction des entités annotées
    if "Entity" in cls:
        score = None
        for attr in a.get("attrs", []):
            if attr.get("label") == "score":
                score = round(attr["value"], 4)
                break

        ents_by_uid[a["uid"]] = {
            "uid": a["uid"],
            "label": a["label"].strip(),
            "text": a["text"].strip(),
            "start": a["spans"][0]["start"],
            "end": a["spans"][0]["end"],
            "score": score,
        }

    # Extraction des relations pertinentes
    elif "Relation" in cls:
        if a["label"] in REL_LINK:
            relations.append({
                "type": a["label"],
                "source_id": a["source_id"],
                "target_id": a["target_id"],
            })

print(f"Entités chargées : {len(ents_by_uid)}")
print(f"Relations filtrées : {len(relations)}")

Entités chargées : 173
Relations filtrées : 100


In [2]:
# Détection des médicaments / classes
meds = sorted(
    [e for e in ents_by_uid.values() if e["label"] in MED_LABELS],
    key=lambda x: x["start"]
)

med_links = defaultdict(list)


# Association médicament -> attributs
for rel in relations:
    src = ents_by_uid.get(rel["source_id"])
    tgt = ents_by_uid.get(rel["target_id"])

    if not src or not tgt:
        continue

    # Si la cible est un médicament et la source un attribut
    if tgt["label"] in MED_LABELS and src["label"] not in MED_LABELS:
        med_links[tgt["uid"]].append(src["uid"])

    # Si la source est un médicament et la cible un attribut
    elif src["label"] in MED_LABELS and tgt["label"] not in MED_LABELS:
        med_links[src["uid"]].append(tgt["uid"])

print(f"Médicaments détectés : {len(meds)}")
print(f"Médicaments avec attributs : {sum(1 for m in meds if med_links.get(m['uid']))}")

Médicaments détectés : 43
Médicaments avec attributs : 39


In [3]:
# Construction des lignes du CSV final
rows = []

for med in meds:
    linked = sorted(
        [ents_by_uid[uid] for uid in med_links.get(med["uid"], []) if uid in ents_by_uid],
        key=lambda x: x["start"]
    )

    # Récupération des valeurs par type d'attribut
    def get_vals(label_set):
        vals = list(dict.fromkeys(
            e["text"] for e in linked if e["label"] in label_set
        ))
        return vals if vals else [""]

    dosages = get_vals(DOSAGE_LABELS)
    frequences = get_vals(FREQ_LABELS)
    voies = get_vals(ROUTE_LABELS)
    dates = get_vals(DATE_LABELS)
    dates_rel = get_vals(DATE_REL_LABELS)
    contextes = get_vals(COND_LABELS)

    # Produit cartésien pour générer une ligne par combinaison d'attributs
    seen = set()
    for (d, f, v, dt, dr, c) in itertools.product(
        dosages, frequences, voies, dates, dates_rel, contextes
    ):
        key = (med["text"], med["start"], d, f, v, dt, dr, c)
        if key in seen:
            continue
        seen.add(key)

        rows.append({
            "Medicament": med["text"],
            "Label_type": med["label"],
            "Dosage": d,
            "Frequence": f,
            "Voie_administration": v,
            "Date": dt,
            "Date_relative": dr,
            "Contexte": c,
            "Position_texte": med["start"],
            "Score": med["score"],
        })


# Export CSV
df = pd.DataFrame(rows, columns=[
    "Medicament", "Label_type", "Dosage", "Frequence",
    "Voie_administration", "Date", "Date_relative",
    "Contexte", "Position_texte", "Score"
])

df.to_csv(OUTPUT_CSV, index=True, index_label="ID", encoding="utf-8-sig")
print(f"{OUTPUT_CSV} — lignes: {len(df)}")

annot_medkit.csv — lignes: 57
